# Resume the existing paired H1 experiment

This notebook resumes saved progress. It has no preprocessing, preflight or new-experiment start cells.

Before stopping the old session, download and retain the full `paired-resume-1790983719.zip` (not the reports ZIP). Upload it privately and attach it to this notebook, together with the SAME `domain-shift-paired` code and `stage1-online-inputs` data. Keep earlier archives locally. Attach only one full resume archive at a time.

Select T4 x2 and enable Internet. Run the five code cells below in order. Only Cell 2 needs your time-accounting entries. Do not add any earlier chat workaround cells. Python recovery installs an isolated runtime; it does not restart the notebook kernel or alter the frozen source code.

Your latest reported state was RMS awaiting an external prefix backup and Taper-minus paused at prefix step 8904. Importing this externally attached archive verifies the backup and preserves both positions. A later prefix boundary can legitimately pause both workers again for a new external backup.


## Cell 1 - Find the unchanged code bundle


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, tempfile, time
inputs = Path('/kaggle/input')
roots = sorted({p.parent.parent.parent for p in inputs.rglob('paired.py')
                if p.parent.name == 'domain_shift_forgetting' and p.parent.parent.name == 'src'})
if roots:
    assert len(roots) == 1, f'Attach exactly one stage1 code bundle: {roots}'
    code = roots[0]
else:
    bundles = list(inputs.rglob('domain-shift-paired.zip'))
    assert len(bundles) == 1, 'Attach domain-shift-paired.zip as a private dataset'
    code = Path(tempfile.mkdtemp(prefix='stage1-code-'))
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (code / name).resolve().is_relative_to(code.resolve())
        z.extractall(code)
env = os.environ | {'PYTHONPATH': str(code / 'src'), 'CUBLAS_WORKSPACE_CONFIG': ':4096:8'}
def cli(module, *args):
    import signal
    command = [sys.executable, '-m', 'domain_shift_forgetting.' + module, *map(str, args)]
    process = subprocess.Popen(command, env=env)
    try:
        result = process.wait()
    except KeyboardInterrupt:
        # Give the coordinator time to stop and join BOTH workers before export.
        if process.poll() is None:
            try:
                process.send_signal(signal.SIGTERM)
            except ProcessLookupError:
                pass
        try:
            process.wait(timeout=300)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            print('Coordinator did not close safely. Full export will refuse active writers; preserve prior durable archive.')
        raise
    if result:
        raise subprocess.CalledProcessError(result, command)
print('Code:', code)


## Cell 2 - Enter current time and quota values


In [ ]:
# The CPU coordinator starts two subprocesses, each seeing exactly its assigned GPU.
env['CUDA_VISIBLE_DEVICES'] = ''
MODE = 'resume'
SESSION_REMAINING_MINUTES = None
WEEKLY_GPU_HOURS_REMAINING = None
# Last successful chunk printed this cumulative external value.
PREVIOUS_EXTERNAL_HOURS = 10.0084886075725
# Enter GPU-session minutes spent exporting, idle, or setting up AFTER that chunk
# ended and BEFORE this settings cell. Exclude stopped/CPU-only time and the chunk itself.
ADDITIONAL_EXTERNAL_MINUTES = None
assert isinstance(ADDITIONAL_EXTERNAL_MINUTES, (int, float)) and ADDITIONAL_EXTERNAL_MINUTES >= 0, 'Fill additional external minutes.'
EXTERNAL_NOTEBOOK_HOURS = PREVIOUS_EXTERNAL_HOURS + ADDITIONAL_EXTERNAL_MINUTES / 60
RUN_PRIMARY = True
RECOVER_UNCLEAN = False  # only after confirming the previous session/worker ended
assert MODE in ('preflight', 'start', 'resume')
assert all(isinstance(x, (int, float)) and x >= 0 for x in
           (SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING, EXTERNAL_NOTEBOOK_HOURS)), 'Fill all three usage values.'
settings_time = time.monotonic()
remaining_at_entry = min(SESSION_REMAINING_MINUTES, WEEKLY_GPU_HOURS_REMAINING * 60)
def remaining():
    value = remaining_at_entry - (time.monotonic() - settings_time) / 60
    assert value > 10, 'Insufficient time for a safe chunk and export reserve.'
    return value
scratch = Path(tempfile.mkdtemp(prefix='paired-'))
policy = code / 'configs/kaggle-stage1-paired.json'
online_roots = [p.parent for p in inputs.rglob('manifest.json') if p.parent.name == 'online'
                and json.loads(p.read_text()).get('schema') == 'pilot-arrays-v1']
if not online_roots:
    bundles = list(inputs.rglob('stage1-online-inputs.zip'))
    assert len(bundles) == 1, 'Attach your existing online inputs.'
    target = scratch / 'inputs'
    with zipfile.ZipFile(bundles[0]) as z:
        for name in z.namelist():
            assert (target / name).resolve().is_relative_to(target.resolve())
        z.extractall(target)
    online_roots = [target / 'online']
assert len(online_roots) == 1, 'Attach exactly one online corpus.'
data = online_roots[0]
orders, upstream = data.parent / 'orders', data.parent / 'upstream'
root = scratch / 'run'
print('Mode:', MODE, 'data:', data, 'temporary run:', root)
print('Policy:', json.loads(policy.read_text())['protocol'], '| cap: 50 elapsed notebook hours')


## Cell 3 - Restore and verify the original Python runtime


In [ ]:
# Select the full saved run attached under /kaggle/input, including extracted ZIPs.
archives = [p for p in inputs.rglob('paired-resume-*.zip') if 'reports' not in p.name]
folders = [p.parent for p in inputs.rglob('archive-receipt.json')
           if (p.parent / 'freeze.json').exists()
           and json.loads((p.parent / 'freeze.json').read_text()).get('layout') == 'paired-v1']
candidates = archives if archives else folders
assert len(candidates) == 1, f'Attach exactly one FULL paired resume archive: {candidates}'
source = candidates[0]
print('Resume source:', source)
if source.is_dir():
    saved = json.loads((source / 'freeze.json').read_text())
    receipt = json.loads((source / 'archive-receipt.json').read_text())
    assert not (source / 'failure.json').exists(), 'Use the successful paused archive, not the failed-launch archive.'
else:
    with zipfile.ZipFile(source) as z:
        saved = json.loads(z.read('freeze.json'))
        receipt = json.loads(z.read('archive-receipt.json'))
        assert 'failure.json' not in z.namelist(), 'Use the successful paused archive, not the failed-launch archive.'
assert receipt.get('kind') == 'resume' and saved.get('admitted') is True
import signal
for condition in ('RMS', 'Taper-minus'):
    r = saved['runtimes'][condition]
    assert (r['python'], r['numpy'], r['torch']) == ('3.12.13', '2.0.2', '2.10.0+cu128'), 'Unexpected saved runtime; stop and report it.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'uv'], check=True)
install_env = os.environ | {
    'UV_PYTHON_INSTALL_DIR': '/tmp/h1-managed-python',
    'UV_CACHE_DIR': '/tmp/h1-uv-cache',
}
uv = [sys.executable, '-m', 'uv', '--no-config']
subprocess.run(uv + ['python', 'install', '3.12.13'], env=install_env, check=True)
venv = Path('/tmp/h1-runtime-py31213')
RUN_PYTHON = str(venv / 'bin/python')
if not Path(RUN_PYTHON).exists():
    subprocess.run(uv + ['venv', '--python', '3.12.13', str(venv)], env=install_env, check=True)
subprocess.run(uv + ['pip', 'install', '--python', RUN_PYTHON,
    'torch==2.10.0+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128'],
    env=install_env, check=True)
subprocess.run(uv + ['pip', 'install', '--python', RUN_PYTHON,
    'numpy==2.0.2', 'pytest==8.4.2', 'datasketch==2.0.0'], env=install_env, check=True)

# Check the exact identity on both GPUs BEFORE changing the training launcher.
probe = r'''
import json, sys, zipfile, torch
from domain_shift_forgetting.pilot_runner import runtime_identity, source_identity
from pathlib import Path
p = Path(sys.argv[1])
member = 'workers/' + sys.argv[2] + '/freeze.json'
if p.is_dir():
    frozen = json.loads((p / member).read_text())
else:
    with zipfile.ZipFile(p) as z:
        frozen = json.loads(z.read(member))
expected = frozen['runtimes'][sys.argv[2]]
actual = runtime_identity(torch.device('cuda'))
differences = {k: {'saved': expected.get(k), 'current': actual.get(k)}
               for k in set(expected) | set(actual) if expected.get(k) != actual.get(k)}
assert frozen['code'] == source_identity(), 'Source code differs from the saved run.'
assert not differences, json.dumps(differences, indent=2)
print(sys.argv[2], 'original runtime matched:', actual, flush=True)
'''
for condition, gpu in [('RMS', '0'), ('Taper-minus', '1')]:
    subprocess.run([RUN_PYTHON, '-c', probe, str(source), condition],
                   env=env | {'CUDA_VISIBLE_DEVICES': gpu}, check=True)

def cli(module, *args):
    command = [RUN_PYTHON, '-m', 'domain_shift_forgetting.' + module, *map(str, args)]
    process = subprocess.Popen(command, env=env)
    try:
        result = process.wait()
    except KeyboardInterrupt:
        if process.poll() is None:
            try:
                process.send_signal(signal.SIGTERM)
            except ProcessLookupError:
                pass
        try:
            process.wait(timeout=300)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
            print('Coordinator did not close safely; preserve the prior archive.')
        raise
    if result:
        raise subprocess.CalledProcessError(result, command)

print('Both GPU runtimes match. Run Cell 4 next.')


## Cell 4 - Import your saved training progress


In [ ]:
# Verified import registers the externally saved prefixes for BOTH workers.
root = Path(tempfile.mkdtemp(prefix='paired-verified-resume-')) / 'run'
cli('paired', 'import', '--archive', source, '--root', root)
for condition in ('RMS', 'Taper-minus'):
    print(condition, (root / 'workers' / condition / 'status.json').read_text())
print('Restored saved progress and registered the external backup. Run Cell 5 next.')


## Cell 5 - Continue training and export the next checkpoint


In [ ]:
if MODE in ('start', 'resume'):
    # Account for this invocation's import, verification and freeze overhead before starting the ledger.
    external_at_run = EXTERNAL_NOTEBOOK_HOURS + (time.monotonic() - settings_time) / 3600
    try:
        cli('paired', 'run', '--root', root, '--data', data, '--orders', orders,
            '--remaining-minutes', remaining(), '--external-notebook-hours', external_at_run,
            '--recover-unclean', int(RECOVER_UNCLEAN))
    finally:
        stamp = str(int(time.time()))
        if (root / 'freeze.json').exists():
            try:
                cli('paired', 'report', '--root', root)
            finally:
                cli('paired', 'export', '--root', root, '--output', f'/kaggle/working/paired-resume-{stamp}.zip')
                cli('paired', 'export', '--root', root, '--reports-only', '--output', f'/kaggle/working/paired-reports-{stamp}.zip')
    print((root / 'status.json').read_text())
    print((root / 'decision-report.txt').read_text())
    ledger = json.loads((root / 'notebook-ledger.json').read_text())
    used = ledger['external_gpu_hours'] + sum(s['charged_seconds'] for s in ledger['sessions']) / 3600
    worker_hours = sum(s.get('summed_worker_seconds', 0) for s in ledger['sessions']) / 3600
    print(f'Cumulative experiment notebook hours: {used:.3f} / 50; recorded worker hours: {worker_hours:.3f}')
    print('External hours recorded at run start:', external_at_run)
    print('Save the FULL paired archive before stopping. Next session: MODE=resume. Keep all older archives.')


After Cell 5 ends cleanly, save the NEW full `paired-resume-*.zip` and the small `paired-reports-*.zip`. Send the small report for review. Keep every full archive. For the next session attach only the latest full archive and use this same notebook, updating Cell 2's `PREVIOUS_EXTERNAL_HOURS` to the value printed at the end of the last successful chunk and adding subsequent external minutes. Do not use total experiment hours as external hours. An incomplete scientific report is expected until the fixed matrix finishes.

Runtime setup follows [uv's Python version documentation](https://docs.astral.sh/uv/concepts/python-versions/) and [PyTorch's previous-version installation instructions](https://pytorch.org/get-started/previous-versions/).
